In [ ]:
from datetime import datetime, timedelta
from shapely.geometry import Polygon, Point
import numpy as np
import requests
import pandas as pd
from shapely.geometry import Polygon
from xml.etree import ElementTree as ET
from shapely.geometry import Polygon
import os


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


STEP1 download of data

In [ ]:
import os
import requests
import pandas as pd


# -----------------------------
# Copernicus Data Space API tools
# -----------------------------

def get_access_and_refresh_token(username, password):
    """
    Retrieve access and refresh tokens from Copernicus Data Space.
    """
    url = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"
    data = {
        "grant_type": "password",
        "username": username,
        "password": password,
        "client_id": "cdse-public",
    }

    response = requests.post(url, data=data)
    response.raise_for_status()
    tokens = response.json()

    return tokens["access_token"], tokens["refresh_token"]


def refresh_access_token(refresh_token):
    """
    Refresh the access token using the refresh token.
    """
    url = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"
    data = {
        "grant_type": "refresh_token",
        "refresh_token": refresh_token,
        "client_id": "cdse-public",
    }

    response = requests.post(url, data=data)
    response.raise_for_status()

    return response.json()["access_token"]


def make_api_request(url, method="GET", data=None, headers=None):
    """
    Make an authenticated API request.
    """
    global access_token
    global refresh_token

    if headers is None:
        headers = {"Authorization": f"Bearer {access_token}"}

    response = requests.request(method, url, json=data, headers=headers)

    if response.status_code in [401, 403]:
        access_token = refresh_access_token(refresh_token)
        headers["Authorization"] = f"Bearer {access_token}"
        response = requests.request(method, url, json=data, headers=headers)

    return response


# -----------------------------
# Sentinel-2 query for cloud classification
# -----------------------------

def query_sentinel2_london_data(start_date, end_date, token):
    """
    Query Sentinel-2 L2A data over London for cloud cover classification.

    Parameters
    ----------
    start_date : str
        Start date in YYYY-MM-DD format.
    end_date : str
        End date in YYYY-MM-DD format.
    token : str
        Copernicus Data Space access token.

    Returns
    -------
    pandas.DataFrame
        Table containing Sentinel-2 products.
    """

    all_data = []

    # Study area: London, UK
    # This region contains urban areas, vegetation, water, clouds and clear pixels.
    london_polygon = (
        "POLYGON((-0.35 51.35, 0.15 51.35, 0.15 51.65, -0.35 51.65, -0.35 51.35))"
    )

    filter_string = (
        f"Collection/Name eq 'SENTINEL-2' and "
        f"Attributes/OData.CSC.StringAttribute/any(att:att/Name eq 'productType' and att/Value eq 'S2MSI2A') and "
        f"ContentDate/Start gt {start_date}T00:00:00.000Z and "
        f"ContentDate/Start lt {end_date}T23:59:59.999Z"
    )

    next_url = (
        "https://catalogue.dataspace.copernicus.eu/odata/v1/Products?"
        f"$filter={filter_string} and "
        f"OData.CSC.Intersects(area=geography'SRID=4326;{london_polygon}')&"
        "$orderby=ContentDate/Start desc&"
        "$top=100"
    )

    headers = {"Authorization": f"Bearer {token}"}

    while next_url:
        response = make_api_request(next_url, headers=headers)

        if response.status_code == 200:
            response_json = response.json()
            data = response_json["value"]
            all_data.extend(data)
            next_url = response_json.get("@odata.nextLink")
        else:
            print(f"Error fetching data: {response.status_code} - {response.text}")
            break

    return pd.DataFrame(all_data)


# -----------------------------
# Download tool
# -----------------------------

def download_single_product(product_id, file_name, access_token, download_dir="downloaded_products"):
    """
    Download a single Sentinel product from Copernicus Data Space.

    Parameters
    ----------
    product_id : str
        Product ID from the query result.
    file_name : str
        Output file name.
    access_token : str
        Copernicus access token.
    download_dir : str
        Directory where the zip file will be saved.
    """

    os.makedirs(download_dir, exist_ok=True)

    url = f"https://zipper.dataspace.copernicus.eu/odata/v1/Products({product_id})/$value"

    headers = {"Authorization": f"Bearer {access_token}"}
    session = requests.Session()
    session.headers.update(headers)

    response = session.get(url, stream=True)

    if response.status_code == 200:
        output_file_path = os.path.join(download_dir, file_name + ".zip")

        with open(output_file_path, "wb") as file:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    file.write(chunk)

        print(f"Downloaded: {output_file_path}")
    else:
        print(f"Failed to download product {product_id}. Status Code: {response.status_code}")

In [ ]:
# Copernicus Data Space login
username = "1966292214@qq.com"
password = "Sns-2!x5uvxSGh!"

access_token, refresh_token = get_access_and_refresh_token(username, password)

# Study period selected for cloud classification over London
start_date = "2022-06-01"
end_date = "2022-06-30"

sentinel2_cloud_data = query_sentinel2_london_data(
    start_date,
    end_date,
    access_token
)

# Preview metadata
sentinel2_cloud_data.head()

,@odata.mediaContentType,Id,Name,ContentType,ContentLength,OriginDate,PublicationDate,ModificationDate,Online,EvictionDate,S3Path,Checksum,ContentDate,Footprint,GeoFootprint
0,application/octet-stream,21b879a5-372a-4c9e-bde4-8a76b1731bf5,S2B_MSIL2A_20220630T110629_N0510_R137_T31UCS_2...,application/octet-stream,425871206,2025-02-16T01:55:13.007000Z,2025-03-10T12:32:57.863868Z,2025-03-10T12:32:57.863868Z,True,9999-12-31T23:59:59.999999Z,/eodata/Sentinel-2/MSI/L2A_N0500/2022/06/30/S2...,"[{'Value': 'dffedfb486041b9d465966b7adfddaff',...","{'Start': '2022-06-30T11:06:29.024000Z', 'End'...",geography'SRID=4326;POLYGON ((1.32919118204554...,"{'type': 'Polygon', 'coordinates': [[[1.329191..."
1,application/octet-stream,7127dd77-fed5-42c5-ae63-ecfdc15b8a46,S2B_MSIL2A_20220630T110629_N0510_R137_T30UYB_2...,application/octet-stream,562645372,2025-02-16T01:55:12.655000Z,2025-03-10T12:34:55.845618Z,2025-03-10T12:34:55.845618Z,True,9999-12-31T23:59:59.999999Z,/eodata/Sentinel-2/MSI/L2A_N0500/2022/06/30/S2...,"[{'Value': '5da85227a0ae27a689e05621e6e5129b',...","{'Start': '2022-06-30T11:06:29.024000Z', 'End'...",geography'SRID=4326;POLYGON ((1.29531060399401...,"{'type': 'Polygon', 'coordinates': [[[1.295310..."
2,application/octet-stream,86415f66-a14b-464b-96b9-1dec56ec0ca5,S2B_MSIL2A_20220630T110629_N0510_R137_T30UXC_2...,application/octet-stream,850350244,2025-02-16T01:55:12.451000Z,2025-03-10T12:33:02.996863Z,2025-03-10T12:33:02.996863Z,True,9999-12-31T23:59:59.999999Z,/eodata/Sentinel-2/MSI/L2A_N0500/2022/06/30/S2...,"[{'Value': 'efdabb039e28da9e1055bc3d88c4ad73',...","{'Start': '2022-06-30T11:06:29.024000Z', 'End'...",geography'SRID=4326;POLYGON ((-1.5321158470421...,"{'type': 'Polygon', 'coordinates': [[[-1.53211..."
3,application/octet-stream,8aae1623-6e85-4fdf-944d-e2b0486fe827,S2B_MSIL2A_20220630T110629_N0510_R137_T31UCT_2...,application/octet-stream,634105396,2025-02-16T01:55:13.032000Z,2025-03-10T12:35:46.715314Z,2025-03-10T12:35:46.715314Z,True,9999-12-31T23:59:59.999999Z,/eodata/Sentinel-2/MSI/L2A_N0500/2022/06/30/S2...,"[{'Value': 'eb51a308d3d915ee3414190884ee9ab2',...","{'Start': '2022-06-30T11:06:29.024000Z', 'End'...",geography'SRID=4326;POLYGON ((1.68265764592881...,"{'type': 'Polygon', 'coordinates': [[[1.682657..."
4,application/octet-stream,a76b7dbb-b230-4dd2-bae9-82283659a67c,S2B_MSIL2A_20220630T110629_N0510_R137_T30UYC_2...,application/octet-stream,684060699,2025-02-16T01:55:12.685000Z,2025-03-10T12:36:40.366472Z,2025-03-10T12:36:40.366472Z,True,9999-12-31T23:59:59.999999Z,/eodata/Sentinel-2/MSI/L2A_N0500/2022/06/30/S2...,"[{'Value': '97234b7bb7e8d140ab7f26e496b627e6',...","{'Start': '2022-06-30T11:06:29.024000Z', 'End'...",geography'SRID=4326;POLYGON ((1.48837660367576...,"{'type': 'Polygon', 'coordinates': [[[1.488376..."


In [ ]:
download_dir = "/content/drive/MyDrive/cloud/4"  # Replace with your desired download directory
product_id = sentinel2_cloud_data['Id'][4] # Replace with your desired file id
file_name = sentinel2_cloud_data['Name'][4]# Replace with your desired filename
# Download the single product
download_single_product(product_id, file_name, access_token, download_dir)

Downloaded: /content/drive/MyDrive/cloud/4/S2B_MSIL2A_20220630T110629_N0510_R137_T30UYC_20240629T122536.SAFE.zip


In [ ]:
cd /content/drive/MyDrive/cloud/4

/content/drive/MyDrive/cloud/4


In [ ]:
! unzip /content/drive/MyDrive/cloud/4/S2B_MSIL2A_20220630T110629_N0510_R137_T30UYC_20240629T122536.SAFE.zip



Archive:  /content/drive/MyDrive/cloud/4/S2B_MSIL2A_20220630T110629_N0510_R137_T30UYC_20240629T122536.SAFE.zip
 extracting: S2B_MSIL2A_20220630T110629_N0510_R137_T30UYC_20240629T122536.SAFE/DATASTRIP/DS_S2RP_20240629T122536_S20220630T110828/MTD_DS.xml  
 extracting: S2B_MSIL2A_20220630T110629_N0510_R137_T30UYC_20240629T122536.SAFE/DATASTRIP/DS_S2RP_20240629T122536_S20220630T110828/QI_DATA/FORMAT_CORRECTNESS.xml  
 extracting: S2B_MSIL2A_20220630T110629_N0510_R137_T30UYC_20240629T122536.SAFE/DATASTRIP/DS_S2RP_20240629T122536_S20220630T110828/QI_DATA/GENERAL_QUALITY.xml  
 extracting: S2B_MSIL2A_20220630T110629_N0510_R137_T30UYC_20240629T122536.SAFE/DATASTRIP/DS_S2RP_20240629T122536_S20220630T110828/QI_DATA/GEOMETRIC_QUALITY.xml  
 extracting: S2B_MSIL2A_20220630T110629_N0510_R137_T30UYC_20240629T122536.SAFE/DATASTRIP/DS_S2RP_20240629T122536_S20220630T110828/QI_DATA/RADIOMETRIC_QUALITY.xml  
 extracting: S2B_MSIL2A_20220630T110629_N0510_R137_T30UYC_20240629T122536.SAFE/DATASTRIP/DS_S2RP_